# makemore part 5: a WaveNet, from scratch

This is the *Practice* step of `unit_06_makemore_wavenet.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one cell of stubs followed by a grader cell.
The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the makemore repo or anyone's notebook.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *Python / torch syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

Everything here is a plain Python class with two methods, `__call__` and `parameters()`.
That is the whole "module" interface for this unit; no `torch.nn` until the very end.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

from test_makemore_wavenet import grade

## Given: the data (block_size 8) and the two layers you already wrote in lecture 4

Boilerplate. Read it once, don't edit it.

In [ ]:
words = open('../data/names.txt').read().splitlines()
chars = sorted(set(''.join(words)))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)

block_size = 8  # was 3 in the MLP lectures

def build_dataset(ws):
    X, Y = [], []
    for w in ws:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

import random
random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print(Xtr.shape, Ytr.shape, '| one example:', ''.join(itos[i.item()] for i in Xtr[7]), '-->', itos[Ytr[7].item()])

In [ ]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []

## Milestone 1 — Embedding and Flatten as modules

In lecture 3 the embedding lookup and the `.view` to flatten were two loose lines in the
forward pass. Make them layers with the same two-method interface as `Linear`, so a
container can treat every step of the forward pass the same way.

In [ ]:
class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        """A lookup table of shape (num_embeddings, embedding_dim), random normal init."""
        raise NotImplementedError

    def __call__(self, IX):
        """IX: integer tensor of any shape, e.g. (B,) or (B, T).
        Returns floats of shape IX.shape + (embedding_dim,): each integer replaced by its row."""
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError


class Flatten:
    def __call__(self, x):
        """x: (B, ...) -> (B, everything_else_multiplied). Keeps element order."""
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError

In [ ]:
grade(Embedding, Flatten, upto=1)

## Milestone 2 — Sequential

A container that is itself a layer: same two methods. After this the whole forward pass is
one call and the whole parameter list is one call.

In [ ]:
class Sequential:
    def __init__(self, layers):
        """layers: a list of layer objects. Keep it as self.layers."""
        raise NotImplementedError

    def __call__(self, x):
        """Feed x through every layer in order; return the last output."""
        raise NotImplementedError

    def parameters(self):
        """ONE flat list of every tensor in every layer, in layer order."""
        raise NotImplementedError

In [ ]:
grade(Embedding, Flatten, Sequential, upto=2)

## Milestone 3 — FlattenConsecutive: the idea of this lecture

`Flatten` squashes all 8 context positions into one vector in a single step, and one
`Linear` has to make sense of all of them at once. Instead, fuse *n consecutive* positions
at a time, so the sequence gets shorter and each position wider, level by level.

Before writing: take a tensor of shape `(4, 8, 10)` in the Scratch cell at the bottom, and
work out by hand which elements have to end up next to each other. Then find the one
reshape that does it. If your first attempt scrambles things, the grader will say so.

In [ ]:
class FlattenConsecutive:
    def __init__(self, n):
        """n: how many consecutive positions to fuse into one."""
        self.n = n

    def __call__(self, x):
        """x: (B, T, C) -> (B, T // n, C * n).
        Output position j holds positions [n*j, n*j+1, ..., n*j + n-1] of the input,
        laid side by side in that order.
        If T // n == 1 the middle dim is dropped, giving (B, C * n): a plain batch again."""
        raise NotImplementedError

    def parameters(self):
        raise NotImplementedError

In [ ]:
grade(Embedding, Flatten, Sequential, FlattenConsecutive, upto=3)

## Milestone 4 — BatchNorm1d that survives a 3-D input

Your lecture-4 `BatchNorm1d` took `(B, C)`. Now it will also be handed `(B, T, C)` from the
middle of the hierarchy. `__init__` is given; write `__call__` so it is correct for *both*
input shapes. Nothing crashes if you get this wrong: the grader compares you against
`torch.nn.BatchNorm1d` on both shapes.

**BatchNorm1d(dim)**
- Holds (given): `gamma`, `beta`, `running_mean`, `running_var`, each shape `(dim,)`;
  `eps`, `momentum`, and a `training` flag.
- `__call__(x)`, `x` of shape `(B, C)` or `(B, T, C)` with `C == dim`: returns
  `gamma * (x - mean) / sqrt(var + eps) + beta`, same shape as `x`. In training mode
  `mean`/`var` are this batch's per-channel statistics over every sample and every
  position, and `running_mean`/`running_var` move `momentum` of the way toward them
  (outside the graph, staying shape `(dim,)`). In eval mode `mean`/`var` are the running
  stats. Biased or unbiased variance both pass.
- `parameters()` (given): `[gamma, beta]`.

The grader checks, in order: attributes and parameters → 2-D shape → 2-D values vs
torch → running_mean after one call → 3-D shape → 3-D values vs torch → running_mean
shape and value after a 3-D call → eval mode on 3-D vs torch.

In [ ]:
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        # trained by backprop
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        # running estimates, updated with momentum, used in eval mode
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        """x: (B, C) or (B, T, C); C == dim. Returns the same shape.
        training=True : normalize each channel with this batch's mean/var, then
                        nudge running_mean/running_var toward them by momentum
                        (that update is not part of the graph).
        training=False: normalize with the running stats instead.
        Then scale by gamma and shift by beta."""
        raise NotImplementedError

    def parameters(self):
        return [self.gamma, self.beta]

In [ ]:
grade(Embedding, Flatten, Sequential, FlattenConsecutive, BatchNorm1d, upto=4)

## Milestone 5 — assemble the hierarchy

Put the pieces together for `block_size` 8, and for any other power of two.

**build_wavenet(vocab_size, n_embd, n_hidden, block_size)**
- Returns a `Sequential` whose `layers` are:
  `Embedding(vocab_size, n_embd)`, then one level per halving of the sequence
  (`log2(block_size)` levels), each level being
  `FlattenConsecutive(2)`, `Linear(fan_in, n_hidden)`, `BatchNorm1d(n_hidden)`, `Tanh()`,
  then `Linear(n_hidden, vocab_size)`.
- `fan_in` of each Linear is the width of what arrives from the `FlattenConsecutive(2)`
  before it. A Linear that feeds a BatchNorm1d has no bias.
- The last Linear's weight is scaled by `0.1` at init, so the first loss is near
  `ln(vocab_size)`.
- On a `(B, block_size)` index batch the model returns logits of shape `(B, vocab_size)`.

The grader checks, in order: returns a Sequential → final shape `(5, 27)` → no layer
flattens all 8 positions at once → sequence dim goes 8 → 4 → 2 → three BatchNorm1d and
three Tanh → each BatchNorm1d sits between a Linear and a Tanh → exact parameter count →
last-layer weight std < 0.05 → initial loss near ln(27) → every parameter gets a
gradient → `block_size=4` also builds with the right shape and count.

In [ ]:
def build_wavenet(vocab_size, n_embd, n_hidden, block_size):
    """Return a Sequential:
         Embedding(vocab_size, n_embd)
         log2(block_size) times: FlattenConsecutive(2), Linear(?, n_hidden), BatchNorm1d(n_hidden), Tanh()
         Linear(n_hidden, vocab_size)
    Shrink the last Linear's weight by 0.1 at init so the net starts unconfident
    (first loss near ln(vocab_size)). Every Linear's fan_in must match what arrives."""
    raise NotImplementedError

In [ ]:
grade(Embedding, Flatten, Sequential, FlattenConsecutive, BatchNorm1d, build_wavenet, upto=5)

## Milestone 6 — it learns

No new code. The grader builds `build_wavenet(27, 10, 68, 8)`, trains it for 2500 steps
(a couple of seconds), switches every layer to eval mode (`layer.training = False`), and
measures the dev loss.

The grader checks, in order: dev loss after 2500 steps < 2.45. It prints the before and
after values either way. Predict the number before you run it.

In [ ]:
grade(Embedding, Flatten, Sequential, FlattenConsecutive, BatchNorm1d, build_wavenet, upto=6)

## Given: a training loop, the smoothed loss plot, and sampling

Boilerplate for exploring. `train` is the lecture's loop; the plot uses the lecture's
trick of averaging the per-step losses in chunks instead of plotting the raw noise.
Try it with `n_embd=24, n_hidden=128` and more steps once milestone 6 is green.

In [ ]:
def train(model, steps=5000, batch_size=32, lr=0.1, lr_decay_at=0.75, seed=42):
    g = torch.Generator().manual_seed(seed)
    params = model.parameters()
    for p in params:
        p.requires_grad = True
    lossi = []
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
        logits = model(Xtr[ix])
        loss = F.cross_entropy(logits, Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        step_lr = lr if i < int(lr_decay_at * steps) else lr / 10
        for p in params:
            p.data += -step_lr * p.grad
        lossi.append(loss.log10().item())
        if i % 1000 == 0:
            print(f'{i:7d}/{steps:7d}: {loss.item():.4f}')
    return lossi


def plot_loss(lossi, chunk=200):
    """Average the log10 losses in chunks so the curve is readable."""
    n = len(lossi) // chunk * chunk
    plt.plot(torch.tensor(lossi[:n]).view(-1, chunk).mean(1))
    plt.xlabel(f'step / {chunk}'); plt.ylabel('log10 loss')


def set_eval(model, flag=True):
    for layer in model.layers:
        layer.training = not flag


@torch.no_grad()
def split_loss(model, split):
    x, y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    set_eval(model)
    loss = F.cross_entropy(model(x), y)
    set_eval(model, False)
    print(split, f'{loss.item():.4f}')


@torch.no_grad()
def sample(model, n=20, seed=2147483647 + 10):
    g = torch.Generator().manual_seed(seed)
    set_eval(model)
    out = []
    for _ in range(n):
        chars_out, context = [], [0] * block_size
        while True:
            logits = model(torch.tensor([context]))
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            if ix == 0:
                break
            chars_out.append(itos[ix])
        out.append(''.join(chars_out))
    set_eval(model, False)
    return out

In [ ]:
torch.manual_seed(42)
model = build_wavenet(vocab_size, n_embd=10, n_hidden=68, block_size=block_size)
print(sum(p.numel() for p in model.parameters()), 'parameters')
lossi = train(model, steps=3000)
plot_loss(lossi)
split_loss(model, 'train'); split_loss(model, 'dev')
print(sample(model, 10))

## Milestone 7 (stretch) — the Linear was a convolution all along

Look at one level: `FlattenConsecutive(2)` followed by `Linear(2*C, H)`. For every window
of two positions it applies the *same* weight matrix. That is exactly a 1-D convolution
with `kernel_size=2, stride=2`, `in_channels=C, out_channels=H`, run over the sequence
axis: `torch.nn.Conv1d(C, H, kernel_size=2, stride=2)`. The whole `build_wavenet` is
three such convolutions with a Tanh between them. Real WaveNet uses `stride=1` with a
growing `dilation` (1, 2, 4, ...) so that *every* position, not just the last one, gets a
prediction, and all of those predictions are trained at once. Same weights, same tree of
influence, far fewer wasted forward passes.

**linear_as_conv(W, n)**
- `W`: shape `(n * C, H)`, the weight of a Linear that follows `FlattenConsecutive(n)`.
  Row `k*C + c` of `W` is the weight for channel `c` of the `k`-th position in the window.
- Returns a tensor of shape `(H, C, n)`, the layout `F.conv1d` wants for
  `(out_channels, in_channels, kernel_size)`, such that
  `F.conv1d(x.transpose(1, 2), w, stride=n).transpose(1, 2)` equals
  `FlattenConsecutive(n)(x) @ W` for any `x` of shape `(B, T, C)`.

The grader checks, in order, for `(n, C, H) = (2, 3, 5)` then `(3, 4, 2)`: output shape
`(H, C, n)` → conv1d with that weight reproduces the Linear numerically.

The final grade cell below passes `skip=(7,)` so you can finish the unit without this one. Nothing after milestone 7 depends on `linear_as_conv`; drop the `skip=` to grade it once it exists.

In [ ]:
def linear_as_conv(W, n):
    """W: (n * C, H), the weight of a Linear that follows FlattenConsecutive(n).
    Return the equivalent conv weight of shape (H, C, n) for F.conv1d(x_BCT, w, stride=n),
    so that conv1d(x.transpose(1, 2), w, stride=n).transpose(1, 2) == FlattenConsecutive(n)(x) @ W.
    Row k*C + c of W is the weight for channel c of the k-th position in the window."""
    raise NotImplementedError

In [ ]:
grade(Embedding, Flatten, Sequential, FlattenConsecutive, BatchNorm1d, build_wavenet, linear_as_conv, skip=(7,))

## Your own training loop

The `train` helper above is the lecture's. Now write one from memory, on the same data,
with the bigger model from the lecture (`n_embd=24, n_hidden=128`). Include: batching,
the loss, zeroing grads, the update, a learning-rate drop, and the eval-mode switch
before you measure the dev loss. Compare your dev loss to the given helper's. If the loss
does something weird, ask.

In [ ]:
torch.manual_seed(42)
model = build_wavenet(vocab_size, n_embd=24, n_hidden=128, block_size=block_size)

for step in range(10000):
    ...

## Scratch

Space to poke at things. `x = torch.randn(4, 8, 10)` and try reshapes; `model.layers[i].out.shape`
after a forward pass shows every intermediate shape.